#   Exploracion de Datos

## Dataset de Produccion

### `df_produccion` — Estimaciones agrícolas (MAGyP)
 
| Columna | Tipo | Descripción |
|---|---|---|
| `cultivo` | texto | Cultivo relevado (soja, maíz, trigo, etc.). Algunos tienen variantes `1ra`/`2da`/`total` |
| `anio` | entero | Año calendario de inicio de la campaña |
| `campania` | texto | Campaña agrícola (ej: `2023/2024`), abarca dos años calendario |
| `provincia` | texto | Nombre de la provincia |
| `provincia_id` | decimal | Código INDEC de la provincia (convertir a entero) |
| `departamento` | texto | Nombre del departamento/partido dentro de la provincia |
| `departamento_id` | decimal | Código INDEC del departamento (convertir a entero) |
| `superficie_sembrada_ha` | entero | Hectáreas sembradas |
| `superficie_cosechada_ha` | entero | Hectáreas efectivamente cosechadas (puede ser menor a la sembrada por pérdidas) |
| `produccion_tm` | entero | Producción total en toneladas |
| `rendimiento_kgxha` | entero | Rendimiento: kilos producidos por hectárea cosechada |
 
---

In [3]:
import pandas as pd

df_produccion = pd.read_csv("../data/raw/estimaciones-agricolas.csv")

df_produccion.head()

,cultivo,anio,campania,provincia,provincia_id,departamento,departamento_id,superficie_sembrada_ha,superficie_cosechada_ha,produccion_tm,rendimiento_kgxha
0,ajo,1969,1969/1970,Buenos Aires,6.0,25 de Mayo,6854.0,3,3,10,3333
1,ajo,1969,1969/1970,Buenos Aires,6.0,Adolfo Gonzales Chaves,6014.0,15,15,82,5467
2,ajo,1969,1969/1970,Buenos Aires,6.0,Almirante Brown,6028.0,2,2,8,4000
3,ajo,1969,1969/1970,Buenos Aires,6.0,Balcarce,6063.0,450,450,2025,4500
4,ajo,1969,1969/1970,Buenos Aires,6.0,Cañuelas,6134.0,2,2,7,3500


In [4]:
df_produccion.shape

(160499, 11)

El dataset tiene 160499 filas y 11 columnas

In [5]:
df_produccion.dtypes

cultivo                        str
anio                         int64
campania                       str
provincia                      str
provincia_id               float64
departamento                   str
departamento_id            float64
superficie_sembrada_ha       int64
superficie_cosechada_ha      int64
produccion_tm                int64
rendimiento_kgxha            int64
dtype: object

`provincia_id` y `departamento_id` estan como `float64`(con decimales) en vez de `int64` (numeros enteros), en la limpieza hay que modificarlo para evitar errores en el futuro.

In [6]:
df_produccion['cultivo'].unique()

<StringArray>
[             'ajo',          'algodón',          'alpiste',
            'arroz',           'arveja',            'avena',
           'banana',   'caña de azúcar',          'cártamo',
 'cebada cervecera', 'cebada forrajera',     'cebada total',
    'cebolla total',          'centeno',            'colza',
         'garbanzo',          'girasol',           'jojoba',
          'lenteja',            'limón',             'lino',
             'maíz',        'mandarina',             'maní',
             'mijo',          'naranja',       'papa total',
           'pomelo',    'poroto alubia',     'poroto negro',
     'poroto otros',     'poroto total',         'soja 1ra',
         'soja 2da',       'soja total',            'sorgo',
               'té',    'trigo candeal',      'trigo total',
             'tung',       'yerba mate']
Length: 41, dtype: str

#### cultivos "repetidos"

`soja 1ra` / `soja 2da` / `soja total` → distintas fechas de siembra en la misma campaña. 
`total` = suma de ambas.
`trigo candeal` / `trigo total`, `cebada cervecera` / `cebada forrajera` / `cebada total` → distintas variedades comerciales. `total` = suma de todas.

**Regla:** usar siempre la variante `"<cultivo> total"` para evitar contar producción dos veces.

In [7]:
df_produccion['provincia'].unique()

<StringArray>
[       'Buenos Aires',          'Entre Ríos',             'Formosa',
               'Jujuy',            'La Rioja',             'Mendoza',
           'Catamarca',            'Misiones',               'Chaco',
             'Neuquén',           'Río Negro',               'Salta',
              'Chubut',            'San Juan',             'Córdoba',
            'San Luis',            'Santa Fe', 'Santiago del Estero',
             'Tucumán',          'Santa Cruz',          'Corrientes',
            'La Pampa',                   nan,    'Tierra del Fuego']
Length: 24, dtype: str

El dataset tiene las 23 provincias de la Argentina y una con valor nulo

In [8]:
df_produccion.isnull().sum()

cultivo                    0
anio                       0
campania                   0
provincia                  8
provincia_id               8
departamento               8
departamento_id            8
superficie_sembrada_ha     0
superficie_cosechada_ha    0
produccion_tm              0
rendimiento_kgxha          0
dtype: int64

Tiene 8 valores nulos en 4 columnas: `provincia`, `provincia_id`, `departamento` y `departamento_id`

## Dataset de Deforestacion

### `df_deforestacion` — Pérdida de bosque nativo por provincia (Ministerio de Ambiente)
 
| Columna | Tipo | Descripción |
|---|---|---|
| `provincia` | texto | Nombre de la provincia (tiene espacios extra a limpiar) |
| `categoría_de_conservación` | texto | Categoría OTBN del bosque perdido: `I` (rojo, máxima protección), `II` (amarillo), `III` (verde, menor restricción), `Sin categoría` |
| `superficie_en_hectáreas` | decimal | Hectáreas de bosque nativo perdidas en esa provincia/categoría |
 
**Nota:** este dataset es una foto acumulada (no tiene columna de año), a diferencia del de producción que sí es serie temporal.

In [9]:
df_deforestacion = pd.read_csv("../data/raw/bqe_e_bn_percatpcia_ha", sep=";", encoding="utf-8")

df_deforestacion.head()

,provincia,categoría_de_conservación,superficie_en_hectáreas
0,Buenos Aires,I,2.0
1,Buenos Aires,II,870.0
2,Buenos Aires,III,158.0
3,Buenos Aires,Sin categoría,2.0
4,Catamarca,I,499.0


In [10]:
df_deforestacion.shape

(92, 3)

El dataset tiene 92 filas y 3 columnas

In [11]:
df_deforestacion.dtypes

provincia                        str
categoría_de_conservación        str
superficie_en_hectáreas      float64
dtype: object

In [12]:
df_deforestacion['provincia'].unique() 

<StringArray>
[         'Buenos Aires',            'Catamarca ',                 'Chaco',
                'Chubut',              'Córdoba ',            'Corrientes',
            'Entre Ríos',               'Formosa',                 'Jujuy',
              'La Pampa',              'La Rioja',               'Mendoza',
            'Misiones  ',               'Neuquén',             'Río Negro',
                 'Salta',            'San Juan  ',              'San Luis',
            'Santa Cruz',             'Santa Fe ', 'Santiago del Estero  ',
      'Tierra del Fuego',               'Tucumán']
Length: 23, dtype: str

23 provincias, pero varias tienen espacios en blanco al final: `'Catamarca '`, `'Córdoba '`, 
`'Misiones  '`, `'San Juan  '`, `'Santa Fe '`, `'Santiago del Estero  '`.

**Por qué importa:** al cruzar con el dataset de producción (que sí tiene los nombres limpios), 
`'Córdoba '` y `'Córdoba'` se tratarían como provincias distintas. Hay que aplicar `.str.strip()` 
antes de cualquier `merge` o `JOIN`.

In [13]:
df_deforestacion['categoría_de_conservación'].unique()

<StringArray>
['I', 'II', 'III', 'Sin categoría']
Length: 4, dtype: str

4 categorías: `I`, `II`, `III` (niveles de conservación del bosque, según la Ley de Bosques 
26.331 — I es la de mayor valor de conservación) y `Sin categoría`.

In [14]:
df_deforestacion.isnull().sum()

provincia                    0
categoría_de_conservación    0
superficie_en_hectáreas      1
dtype: int64

1 fila con `superficie_en_hectáreas` vacía. Hay que identificarla antes de decidir si se 
elimina o se completa con 0.